# Advisory parameter selection

These routines benchmark suggestions for one problem; they do not change `UniformFmm` defaults. Performance depends on geometry, particle count, target distribution, CPU, GPU, thread count, backend, and compiler optimisation. Accuracy additionally depends on moments, targets, order, depth, and deterministic sample selection.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cdfmm

rng = np.random.default_rng(13)
sources = rng.uniform(-1, 1, (2000, 3))
targets = rng.uniform(-1, 1, (1000, 3))
moments = rng.normal(size=(2000, 3))


## Performance sweep

The CPU backend is sequential, so measured total time selects the candidate; balance is diagnostic. Partial CUDA overlaps near and far work and uses `max(T_near, T_far)` as its branch-cost heuristic while retaining wall time.


In [ ]:
performance = cdfmm.suggest_depth_for_performance(
    sources, targets, moments, order=6,
    backend=cdfmm.ExecutionBackend.CPU_STATIC,
    candidate_depths=[1, 2, 3, 4], repetitions=3,
)
timing_table = pd.DataFrame(performance["candidates"])
display(timing_table[["depth", "near_seconds", "far_seconds",
                      "balance_ratio", "evaluation_seconds"]])

fig, axis = plt.subplots()
axis.plot(timing_table.depth, timing_table.near_seconds, "o-", label="near")
axis.plot(timing_table.depth, timing_table.far_seconds, "o-", label="far")
axis.plot(timing_table.depth, timing_table.evaluation_seconds, "o-", label="total")
axis.axvline(performance["suggested_depth"], color="black", linestyle="--",
             label="suggested")
axis.set(xlabel="tree depth", ylabel="seconds")
axis.legend();


## Sampled accuracy sweep

The direct reference is computed once for the same deterministic targets. The recommendation is the fastest tested pair meeting the requested sampled RMS relative field error, not the most accurate pair.


In [ ]:
accuracy = cdfmm.suggest_parameters_for_accuracy(
    sources, targets, moments, desired_accuracy=1e-3,
    candidate_orders=[3, 4, 5, 6], candidate_depths=[1, 2, 3, 4],
    sample_size=128, repetitions=3,
)
accuracy_table = pd.DataFrame(accuracy["candidates"])
error_heatmap = accuracy_table.pivot(index="depth", columns="order",
                                     values="rms_relative_error")
display(error_heatmap)
display(accuracy_table[["depth", "order", "rms_relative_error",
                        "evaluation_seconds", "satisfies_accuracy"]])

fig, axis = plt.subplots()
image = axis.imshow(error_heatmap, origin="lower", aspect="auto")
axis.set_xticks(range(len(error_heatmap.columns)), error_heatmap.columns)
axis.set_yticks(range(len(error_heatmap.index)), error_heatmap.index)
axis.set(xlabel="order", ylabel="depth", title="Sampled RMS relative field error")
if accuracy["suggested_order"] >= 0:
    x = list(error_heatmap.columns).index(accuracy["suggested_order"])
    y = list(error_heatmap.index).index(accuracy["suggested_depth"])
    axis.plot(x, y, "wx", markersize=14, markeredgewidth=3)
fig.colorbar(image, ax=axis);


For repeated static problems, run an adviser once during setup, then explicitly assign its returned values to `options.expansion_order` and `options.tree.max_level`. A sampled estimate is not a rigorous global error bound or a guarantee of universal optimality.
